<div style="background:#0B1F4B;padding:22px 26px;border-radius:12px;border:3px solid #C9A227;font-family:'Times New Roman',serif">
<img src="https://raw.githubusercontent.com/drssnaiml1-cell/26UAMPC204-Artificial-Intelligence/main/assets/logo.png" alt="Sanjivani University" width="92" style="float:right;background:#FFFFFF;border:2px solid #C9A227;border-radius:8px;padding:3px;margin-left:14px">
<div style="color:#C9A227;font-size:14px;letter-spacing:2px;font-weight:bold">26UAMPC204 · ARTIFICIAL INTELLIGENCE LAB · SANJIVANI UNIVERSITY</div>
<div style="color:#FFFFFF;font-size:30px;font-weight:bold;margin-top:6px">Experiment 7: Unification and Resolution in First-Order Logic</div>
<div style="color:#F6EBC8;font-size:15px;margin-top:8px"><b>Course Outcome:</b> CO204.3 &nbsp;|&nbsp; <b>Faculty:</b> Dr. S. Satyanarayana, Professor &amp; Advisor, AI &amp; DS</div>
<div style="clear:both"></div></div>

<div style="background:#F6EBC8;padding:12px 18px;border-radius:8px;margin-top:10px;font-family:'Times New Roman',serif;color:#0B1F4B">
<b>🎯 Aim:</b> Implement the unification algorithm (with occurs check) and a resolution refutation prover for first-order clauses; prove classic textbook theorems.</div>

## 📘 Theory

**Unification** finds the most general substitution $\theta$ such that $p\theta=q\theta$. The **occurs check** rejects bindings like $x/f(x)$.
**Resolution:** from clauses $(\ell_1\vee\alpha)$ and $(\neg\ell_2\vee\beta)$ with $\textsc{Unify}(\ell_1,\ell_2)=\theta$ derive $(\alpha\vee\beta)\theta$.
To prove $KB\models q$: add $\neg q$ to the clause set and derive the empty clause $\square$.

## 🧭 Algorithm

1. Represent terms as nested tuples: variables are lowercase strings starting with '?'; f(x) is ('f', x).
2. UNIFY(x, y, θ): if equal return θ; if x is a variable bind it (with occurs check); if compound unify arguments pairwise.
3. Convert KB and negated goal into clauses (sets of literals).
4. Repeatedly resolve pairs of clauses; stop when the empty clause appears (proof) or no new clauses (no proof).

## ⚙️ Setup

In [1]:
# ---- Course style (navy & gold, Times) -------------------------------------
import os, warnings
warnings.filterwarnings("ignore")
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import Image, display

NAVY, NAVY2, GOLD, GOLDL = "#0B1F4B", "#1F3A7A", "#C9A227", "#F6EBC8"
TEAL, CRIMSON, LEAF, MIST, SLATE = "#0E7C86", "#A4243B", "#2E7D32", "#EEF2F8", "#4A5568"
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Liberation Serif", "DejaVu Serif"],
    "axes.edgecolor": SLATE, "axes.titleweight": "bold", "axes.titlecolor": NAVY,
    "axes.titlesize": 14, "axes.labelcolor": NAVY, "figure.dpi": 110,
    "axes.spines.top": False, "axes.spines.right": False,
})
FIG = os.path.join("..", "assets", "figures");    os.makedirs(FIG, exist_ok=True)
ANI = os.path.join("..", "assets", "animations"); os.makedirs(ANI, exist_ok=True)

def save_gif(anim, name, fps=2):
    path = os.path.join(ANI, name)
    anim.save(path, writer=animation.PillowWriter(fps=fps))
    plt.close("all")
    return path
print("Environment ready  ✓  numpy", np.__version__, "| matplotlib", matplotlib.__version__)


Environment ready  ✓  numpy 2.5.3 | matplotlib 3.11.2


## 💻 Implementation and Output

In [2]:
def is_var(t): return isinstance(t, str) and t.startswith("?")
def walk(t, th):
    while is_var(t) and t in th: t = th[t]
    return t
def occurs(v, t, th):
    t = walk(t, th)
    if t == v: return True
    return isinstance(t, tuple) and any(occurs(v, a, th) for a in t[1:])
def unify(x, y, th=None):
    if th is None: th = {}
    x, y = walk(x, th), walk(y, th)
    if x == y: return th
    if is_var(x): return None if occurs(x, y, th) else {**th, x: y}
    if is_var(y): return unify(y, x, th)
    if isinstance(x, tuple) and isinstance(y, tuple) and x[0] == y[0] and len(x) == len(y):
        for a, b in zip(x[1:], y[1:]):
            th = unify(a, b, th)
            if th is None: return None
        return th
    return None
def resolve_term(t, th):
    t = walk(t, th)
    return (t[0],) + tuple(resolve_term(a, th) for a in t[1:]) if isinstance(t, tuple) else t
def show(t):
    if isinstance(t, tuple): return f"{t[0]}({', '.join(show(a) for a in t[1:])})"
    return t.lstrip("?")
def show_th(th): return "{" + ", ".join(f"{k.lstrip('?')}/{show(resolve_term(v, th))}" for k, v in th.items()) + "}" if th is not None else "FAIL"

tests = [(("Knows","John","?x"), ("Knows","John","Jane")),
         (("Knows","John","?x"), ("Knows","?y","Bill")),
         (("Knows","John","?x"), ("Knows","?y",("Mother","?y"))),
         (("Knows","John","?x"), ("Knows","?x","Elizabeth")),
         (("P","?x",("f","?y")), ("P","a",("f",("g","?z")))),
         (("Q","?x"), ("Q",("f","?x")))]
for a, b in tests: print(f"UNIFY({show(a):<22}, {show(b):<26}) = {show_th(unify(a, b))}")

UNIFY(Knows(John, x)        , Knows(John, Jane)         ) = {x/Jane}
UNIFY(Knows(John, x)        , Knows(y, Bill)            ) = {y/John, x/Bill}
UNIFY(Knows(John, x)        , Knows(y, Mother(y))       ) = {y/John, x/Mother(John)}
UNIFY(Knows(John, x)        , Knows(x, Elizabeth)       ) = FAIL
UNIFY(P(x, f(y))            , P(a, f(g(z)))             ) = {x/a, y/g(z)}
UNIFY(Q(x)                  , Q(f(x))                   ) = FAIL


### Resolution refutation prover

In [3]:
import itertools
def neg(l): return l[1] if l[0] == "not" else ("not", l)
def subst_lit(l, th): return ("not", resolve_term(l[1], th)) if l[0] == "not" else resolve_term(l, th)
cid = itertools.count()
def standardise(clause):
    k = next(cid); m = {}
    def rn(t):
        if is_var(t): m.setdefault(t, f"{t}{k}"); return m[t]
        return (t[0],) + tuple(rn(a) for a in t[1:]) if isinstance(t, tuple) else t
    return frozenset(("not", rn(l[1])) if l[0] == "not" else rn(l) for l in clause)
def resolvents(c1, c2):
    out = []
    for l1 in c1:
        for l2 in c2:
            if (l1[0] == "not") != (l2[0] == "not"):
                a, b = (l1[1], l2) if l1[0] == "not" else (l1, l2[1])
                th = unify(a, b)
                if th is not None:
                    new = frozenset(subst_lit(l, th) for l in (c1 - {l1}) | (c2 - {l2}))
                    out.append((new, th, l1, l2))
    return out
def prove(kb, goal, max_iter=2000):
    clauses = [standardise(c) for c in kb] + [standardise([neg(goal)])]
    names = {c: f"C{i+1}" for i, c in enumerate(clauses)}; steps = []; seen = set(clauses)
    for _ in range(max_iter):
        new = []
        for c1, c2 in itertools.combinations(list(seen), 2):
            for r, th, l1, l2 in resolvents(c1, c2):
                if r in seen or r in [n for n, *_ in new]: continue
                new.append((r, c1, c2, th))
                if not r:
                    steps.append((r, c1, c2, th)); return True, steps, names
        if not new: return False, steps, names
        for r, c1, c2, th in sorted(new, key=lambda x: len(x[0]))[:30]:
            seen.add(r); names[r] = f"C{len(names)+1}"; steps.append((r, c1, c2, th))
    return False, steps, names
def cs(c): return " ∨ ".join(("¬" + show(l[1])) if l[0] == "not" else show(l) for l in sorted(c, key=str)) or "□"

# Example 1 — Socrates
KB1 = [[("not", ("Man","?x")), ("Mortal","?x")], [("Man","Socrates")]]
ok, st, nm = prove(KB1, ("Mortal","Socrates"))
print("KB: ∀x Man(x) ⇒ Mortal(x);  Man(Socrates)   Goal: Mortal(Socrates)")
for r, c1, c2, th in st: print(f"   resolve [{cs(c1)}] with [{cs(c2)}]  ⟹  {cs(r)}")
print("Proved!" if ok else "Not proved")

KB: ∀x Man(x) ⇒ Mortal(x);  Man(Socrates)   Goal: Mortal(Socrates)
   resolve [Mortal(x0) ∨ ¬Man(x0)] with [Man(Socrates)]  ⟹  Mortal(Socrates)
   resolve [Mortal(x0) ∨ ¬Man(x0)] with [¬Mortal(Socrates)]  ⟹  ¬Man(Socrates)
   resolve [¬Mortal(Socrates)] with [Mortal(Socrates)]  ⟹  □
Proved!


In [4]:
# Example 2 — Colonel West (Russell & Norvig 9.3)
KB2 = [
 [("not",("American","?x")),("not",("Weapon","?y")),("not",("Sells","?x","?y","?z")),("not",("Hostile","?z")),("Criminal","?x")],
 [("not",("Missile","?x")),("not",("Owns","Nono","?x")),("Sells","West","?x","Nono")],
 [("not",("Missile","?x")),("Weapon","?x")],
 [("not",("Enemy","?x","America")),("Hostile","?x")],
 [("Owns","Nono","M1")],[("Missile","M1")],[("American","West")],[("Enemy","Nono","America")]]
ok, st, nm = prove(KB2, ("Criminal","West"))
print("Goal: Criminal(West) →", "PROVED ✓" if ok else "not proved", f"after generating {len(st)} resolvents")
chain = []
r = st[-1]
print("Final step:", f"[{cs(r[1])}] + [{cs(r[2])}] ⟹ {cs(r[0])}")

Goal: Criminal(West) → PROVED ✓ after generating 100 resolvents
Final step: [¬Sells(West, M1, Nono)] + [Sells(West, M1, Nono)] ⟹ □


In [5]:
# Visualise the Socrates refutation as a proof tree
fig, ax = plt.subplots(figsize=(7.5, 4)); ax.axis("off")
boxes = {"C1": (0.15, 0.85, "¬Man(x) ∨ Mortal(x)"), "C3": (0.62, 0.85, "¬Mortal(Socrates)  [negated goal]"),
         "R1": (0.38, 0.5, "¬Man(Socrates)   θ={x/Socrates}"), "C2": (0.85, 0.5, "Man(Socrates)"), "R2": (0.6, 0.12, "□  (empty clause)")}
for k, (x, y, t) in boxes.items():
    ax.text(x, y, t, ha="center", va="center", fontsize=12, color="white" if k.startswith("R") else NAVY,
            bbox=dict(boxstyle="round,pad=0.5", fc=NAVY if k.startswith("R") else GOLDL, ec=GOLD, lw=2))
for a, b in (("C1","R1"),("C3","R1"),("R1","R2"),("C2","R2")):
    ax.annotate("", xy=boxes[b][:2], xytext=boxes[a][:2], arrowprops=dict(arrowstyle="-|>", color=TEAL, lw=2, shrinkA=18, shrinkB=18))
ax.set_title("Resolution refutation proof of Mortal(Socrates)")
plt.savefig(f"{FIG}/exp07_resolution_tree.png", dpi=130, bbox_inches="tight"); plt.show()

## ✅ Result and Conclusion

<div style="border-left:6px solid #2E7D32;background:#E3F2E4;padding:12px 18px;border-radius:6px;font-family:'Times New Roman',serif;color:#1A202C">
<b style="color:#2E7D32">Conclusion</b><br>The unifier returned the expected MGUs, correctly failed on the clash requiring standardising apart, and the occurs check blocked x/f(x). The resolution prover derived the empty clause for both Mortal(Socrates) and Criminal(West).</div>

## 🎤 Viva Questions

1. What is a most general unifier?
2. Why is the occurs check needed?
3. Why must variables be standardised apart before resolution?
4. What is Skolemisation?
5. Why is resolution called refutation-complete?